# 3D Multi-Object Tracking — AB3DMOT

## 목표
- **AB3DMOT** (IROS 2020) 핵심 알고리즘 직접 구현
  - **3D Kalman Filter**: (x, y, z, θ, l, w, h, vx, vy) 상태 벡터
  - **3D IoU 기반 헝가리안 매칭**
- Phase 2에서 구현한 2D ByteTrack과의 차이 비교
- KITTI 포맷 합성 시퀀스로 MOTA/MOTP 평가

## 핵심 논문
- **AB3DMOT: A Baseline for 3D Multi Object Tracking** (Weng et al., IROS 2020)
- 핵심 아이디어: 2D Tracking의 기본 구조(Kalman + Hungarian)를 3D BEV 공간으로 확장
  - 2D bbox → 3D bbox (x, y, z, l, w, h, θ)
  - 2D IoU → **3D IoU** (BEV 또는 3D 공간)

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import matplotlib.cm as cm
from scipy.optimize import linear_sum_assignment
from scipy.spatial.transform import Rotation
from collections import defaultdict
import random
import math
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
random.seed(42)
print('모듈 로드 완료')

## 1. AB3DMOT 이론

### 2D Tracking vs 3D Tracking

```
2D Tracking (ByteTrack):          3D Tracking (AB3DMOT):
  상태: [x, y, w, h, vx, vy]        상태: [x, y, z, θ, l, w, h, vx, vy]
  매칭: 2D IoU                       매칭: 3D IoU (BEV)
  한계: depth 정보 없음              장점: 3D 공간에서 정확한 위치 추적
```

### 3D Kalman Filter 상태 벡터 (9차원)

```
x = [cx, cy, cz, θ, l, w, h, vx, vy]
     위치(3D)  방향  크기(3D)  속도(2D)

관측: [cx, cy, cz, θ, l, w, h]  (7차원)
속도(vx, vy)는 직접 측정 불가 → 상태 추정으로 계산
```

### BEV 3D IoU 계산
BEV(Bird's Eye View)에서 회전 사각형 간 IoU:
- 두 bbox의 꼭짓점 계산 (회전 고려)
- Sutherland-Hodgman 폴리곤 교집합 알고리즘
- 교집합 면적 / 합집합 면적

In [ ]:
# ────────────────────────────────────────────────────
# BEV 회전 사각형 IoU 구현
# Sutherland-Hodgman 폴리곤 클리핑 알고리즘
# ────────────────────────────────────────────────────

def box_to_corners_bev(cx, cy, l, w, theta):
    """3D bbox → BEV 꼭짓점 4개 (회전 적용)
    cx, cy: BEV 중심
    l, w: 길이(전후), 너비(좌우)
    theta: heading angle (rad)
    """
    # 미회전 꼭짓점 (중심 기준)
    corners = np.array([
        [ l/2,  w/2],
        [-l/2,  w/2],
        [-l/2, -w/2],
        [ l/2, -w/2],
    ])
    # 회전 행렬
    cos, sin = math.cos(theta), math.sin(theta)
    R = np.array([[cos, -sin], [sin, cos]])
    corners = corners @ R.T + np.array([cx, cy])
    return corners


def polygon_clip(poly, clip_edge_start, clip_edge_end):
    """Sutherland-Hodgman: clip poly against a single half-plane"""
    if len(poly) == 0:
        return []
    result = []
    s = poly[-1]
    for e in poly:
        # e가 clip line 안쪽?
        def inside(p):
            return ((clip_edge_end[0] - clip_edge_start[0]) * (p[1] - clip_edge_start[1]) -
                    (clip_edge_end[1] - clip_edge_start[1]) * (p[0] - clip_edge_start[0])) >= 0
        def intersect(a, b, c, d):
            A1 = b[1]-a[1]; B1 = a[0]-b[0]; C1 = A1*a[0]+B1*a[1]
            A2 = d[1]-c[1]; B2 = c[0]-d[0]; C2 = A2*c[0]+B2*c[1]
            det = A1*B2 - A2*B1
            if abs(det) < 1e-10:
                return a
            return ((C1*B2-C2*B1)/det, (A1*C2-A2*C1)/det)
        if inside(e):
            if not inside(s):
                result.append(intersect(s, e, clip_edge_start, clip_edge_end))
            result.append(e)
        elif inside(s):
            result.append(intersect(s, e, clip_edge_start, clip_edge_end))
        s = e
    return result


def polygon_area(pts):
    """Shoelace 공식으로 폴리곤 면적"""
    if len(pts) < 3:
        return 0.0
    pts = np.array(pts)
    n = len(pts)
    area = 0.0
    for i in range(n):
        j = (i+1) % n
        area += pts[i,0] * pts[j,1]
        area -= pts[j,0] * pts[i,1]
    return abs(area) / 2.0


def iou_bev(box1, box2):
    """
    BEV 회전 3D IoU 계산
    box: (cx, cy, cz, theta, l, w, h)
    """
    cx1,cy1,cz1,th1,l1,w1,h1 = box1
    cx2,cy2,cz2,th2,l2,w2,h2 = box2

    # BEV 꼭짓점
    c1 = box_to_corners_bev(cx1, cy1, l1, w1, th1).tolist()
    c2 = box_to_corners_bev(cx2, cy2, l2, w2, th2).tolist()

    # 교집합 폴리곤 (Sutherland-Hodgman)
    inter = list(c1)
    for i in range(len(c2)):
        if not inter:
            break
        inter = polygon_clip(inter, c2[i], c2[(i+1)%len(c2)])

    inter_area = polygon_area(inter)

    # 합집합
    area1 = l1 * w1
    area2 = l2 * w2
    union_area = area1 + area2 - inter_area + 1e-8

    # 높이 방향 겹침 (z축)
    z_min1, z_max1 = cz1 - h1/2, cz1 + h1/2
    z_min2, z_max2 = cz2 - h2/2, cz2 + h2/2
    z_inter = max(0, min(z_max1, z_max2) - max(z_min1, z_min2))

    # 3D IoU: BEV IoU × 높이 겹침 비율
    iou_bev_val = inter_area / union_area
    h_union = max(z_max1, z_max2) - min(z_min1, z_min2) + 1e-8
    iou_3d = iou_bev_val * z_inter / h_union

    return float(iou_3d)


# 테스트
box_a = (0.0, 0.0, 0.0, 0.0,    4.0, 2.0, 1.5)   # 직진 방향
box_b = (0.5, 0.3, 0.0, 0.1,    4.0, 2.0, 1.5)   # 약간 오프셋+회전
box_c = (10.0, 10.0, 0.0, 0.0,  4.0, 2.0, 1.5)   # 멀리 떨어진 박스

print(f'동일 박스 IoU:         {iou_bev(box_a, box_a):.4f}  (≈ 1.0 기대)')
print(f'인접 박스 IoU:         {iou_bev(box_a, box_b):.4f}')
print(f'멀리 떨어진 박스 IoU:  {iou_bev(box_a, box_c):.4f}  (≈ 0.0 기대)')

## 2. 3D Kalman Filter 구현

In [ ]:
class KalmanFilter3D:
    """
    AB3DMOT 3D Kalman Filter

    상태 벡터 x (9차원):
      [cx, cy, cz, theta, l, w, h, vx, vy]
      위치(3) + 방향(1) + 크기(3) + 속도(2)

    관측 벡터 z (7차원):
      [cx, cy, cz, theta, l, w, h]
    """

    def __init__(self, initial_bbox):
        """
        initial_bbox: (cx, cy, cz, theta, l, w, h)
        """
        nx = 9   # 상태 차원
        nz = 7   # 관측 차원

        # ── 상태 전이 행렬 F (등속 운동 모델) ─────────
        # x_{k+1} = F * x_k
        self.F = np.eye(nx)
        self.F[0, 7] = 1.0   # cx += vx
        self.F[1, 8] = 1.0   # cy += vy

        # ── 관측 행렬 H ────────────────────────────
        # z = H * x (위치/방향/크기만 관측, 속도는 관측 불가)
        self.H = np.zeros((nz, nx))
        self.H[:7, :7] = np.eye(7)

        # ── 프로세스 노이즈 Q ──────────────────────
        self.Q = np.eye(nx)
        self.Q[7, 7] = 0.1   # vx 분산 (속도 변화 작음)
        self.Q[8, 8] = 0.1   # vy 분산

        # ── 관측 노이즈 R ──────────────────────────
        self.R = np.eye(nz)
        self.R[3, 3] = 0.1   # theta 노이즈 (각도는 비교적 정확)

        # ── 초기 공분산 P ──────────────────────────
        self.P = np.eye(nx) * 10.0
        self.P[7, 7] = 100.0   # 초기 속도 불확실성 높음
        self.P[8, 8] = 100.0

        # ── 초기 상태 ──────────────────────────────
        self.x = np.zeros(nx)
        self.x[:7] = initial_bbox
        # vx, vy = 0 (초기 속도 모름)

    def predict(self):
        """예측 단계: 이전 상태 → 현재 예측"""
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        return self.x[:7]   # 관측 가능한 7차원 반환

    def update(self, z):
        """갱신 단계: 관측값으로 상태 보정"""
        z = np.array(z)

        # 혁신 (Innovation): 예측과 관측의 차이
        y = z - self.H @ self.x

        # Kalman Gain: K = P H^T (H P H^T + R)^{-1}
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)

        # 상태 갱신
        self.x = self.x + K @ y

        # 공분산 갱신 (Joseph form — 수치 안정성)
        I = np.eye(len(self.x))
        self.P = (I - K @ self.H) @ self.P

        return self.x[:7]

    def get_state(self):
        """현재 추정 bbox (cx, cy, cz, theta, l, w, h)"""
        return tuple(self.x[:7])

    def get_velocity(self):
        """추정 속도 (vx, vy)"""
        return (self.x[7], self.x[8])


# Kalman Filter 동작 확인
kf = KalmanFilter3D((0.0, 0.0, 0.5, 0.0, 4.5, 2.0, 1.5))
print('초기 상태:', [f'{v:.3f}' for v in kf.get_state()])

# 차량이 직진 (x 방향으로 1m/frame)
for t in range(1, 4):
    pred = kf.predict()
    obs  = (t * 1.0, 0.0, 0.5, 0.0, 4.5, 2.0, 1.5)   # 실제 관측
    upd  = kf.update(obs)
    vx, vy = kf.get_velocity()
    print(f't={t}: pred_x={pred[0]:.3f} → obs_x={obs[0]:.3f} → upd_x={upd[0]:.3f} | vx={vx:.3f}')

## 3. AB3DMOT 트래커 구현

In [ ]:
class Track3D:
    """단일 3D 트랙 객체"""
    _id_counter = 1

    def __init__(self, bbox):
        self.id   = Track3D._id_counter
        Track3D._id_counter += 1
        self.kf   = KalmanFilter3D(bbox)
        self.hits  = 1     # 매칭된 횟수
        self.misses = 0    # 미매칭 연속 횟수
        self.age   = 1
        self.state = 'tentative'   # tentative → confirmed → deleted

        self.history = [bbox]   # 궤적 저장

    def predict(self):
        self.age += 1
        return self.kf.predict()

    def update(self, bbox):
        self.kf.update(bbox)
        self.hits   += 1
        self.misses  = 0
        self.history.append(self.kf.get_state())

        # 3 hit 이상이면 confirmed
        if self.hits >= 3:
            self.state = 'confirmed'

    def mark_missed(self):
        self.misses += 1

    def get_bbox(self):
        return self.kf.get_state()


class AB3DMOT:
    """
    AB3DMOT: 3D Multi-Object Tracker

    파이프라인:
      1. 모든 기존 트랙 Kalman predict
      2. 3D IoU Cost Matrix 계산
      3. 헝가리안 알고리즘으로 매칭
      4. 매칭된 트랙 Kalman update
      5. 미매칭 detection → 신규 트랙 생성
      6. 미매칭 트랙 → miss 카운트, 임계 초과 시 삭제
    """

    def __init__(self,
                 iou_threshold=0.01,   # 3D IoU 매칭 임계값 (3D는 2D보다 낮게)
                 max_misses=3):        # 연속 미매칭 허용 횟수
        self.tracks      = []
        self.iou_thresh  = iou_threshold
        self.max_misses  = max_misses
        self.frame_count = 0

        Track3D._id_counter = 1   # ID 리셋

    def update(self, detections):
        """
        detections: list of (cx, cy, cz, theta, l, w, h)
        returns: list of (cx, cy, cz, theta, l, w, h, track_id)
        """
        self.frame_count += 1
        results = []

        # ── 1. Predict ─────────────────────────────
        for trk in self.tracks:
            trk.predict()

        if len(detections) == 0:
            for trk in self.tracks:
                trk.mark_missed()
            self._remove_dead_tracks()
            return []

        # ── 2. Cost Matrix (3D IoU) ─────────────────
        n_det = len(detections)
        n_trk = len(self.tracks)

        if n_trk > 0:
            iou_matrix = np.zeros((n_trk, n_det))
            for ti, trk in enumerate(self.tracks):
                trk_bbox = trk.get_bbox()
                for di, det in enumerate(detections):
                    iou_matrix[ti, di] = iou_bev(trk_bbox, det)

            # ── 3. Hungarian Matching ─────────────────
            cost = 1.0 - iou_matrix   # 최소화 → IoU 최대화
            row_ind, col_ind = linear_sum_assignment(cost)

            matched_trk = set()
            matched_det = set()

            for ti, di in zip(row_ind, col_ind):
                if iou_matrix[ti, di] >= self.iou_thresh:
                    matched_trk.add(ti)
                    matched_det.add(di)
                    # ── 4. Update matched tracks ────────
                    self.tracks[ti].update(detections[di])

            # ── 5. 미매칭 트랙 miss 처리 ───────────────
            for ti, trk in enumerate(self.tracks):
                if ti not in matched_trk:
                    trk.mark_missed()

            # ── 6. 신규 트랙 생성 ──────────────────────
            for di, det in enumerate(detections):
                if di not in matched_det:
                    self.tracks.append(Track3D(det))
        else:
            # 트랙 없음 → 모든 detection을 신규 트랙으로
            for det in detections:
                self.tracks.append(Track3D(det))

        self._remove_dead_tracks()

        # Confirmed 트랙만 반환
        for trk in self.tracks:
            if trk.state == 'confirmed':
                bbox = trk.get_bbox()
                results.append((*bbox, trk.id))

        return results

    def _remove_dead_tracks(self):
        self.tracks = [
            trk for trk in self.tracks
            if trk.misses <= self.max_misses
        ]


print('AB3DMOT 클래스 준비 완료')

## 4. 합성 KITTI 포맷 시퀀스 생성

In [ ]:
def generate_kitti_sequence(n_frames=60, n_vehicles=8, seed=42):
    """
    자율주행 시나리오 합성 시퀀스 생성 (KITTI 포맷)

    Returns:
        gt_sequence: list of frames, 각 frame = list of (cx,cy,cz,theta,l,w,h,track_id)
        det_sequence: GT에 noise 추가한 detection (실제 3D Detector 출력 모사)
    """
    np.random.seed(seed)
    random.seed(seed)

    # 차량 초기 상태 설정
    vehicles = []
    for vid in range(n_vehicles):
        # 각 차량: 초기 위치, 속도, 방향, 크기
        cx = random.uniform(-20, 20)
        cy = random.uniform(0, 40)
        cz = 0.75   # 지면에서 0.75m (차량 중심)
        theta = random.uniform(-0.3, 0.3)   # 거의 직진

        # 속도: 대부분 y 방향(전진), 약간 x 방향
        vx = random.uniform(-0.3, 0.3)
        vy = random.uniform(0.3, 1.5)   # 다양한 속도

        # 크기: 자동차 기준
        l = random.uniform(4.0, 5.0)
        w = random.uniform(1.8, 2.2)
        h = 1.5

        # 차량이 시야에 들어왔다 나가는 시점 다양화
        start_frame = random.randint(0, 10)
        end_frame   = random.randint(40, n_frames)

        vehicles.append({
            'id': vid+1,
            'cx': cx, 'cy': cy, 'cz': cz,
            'theta': theta, 'vx': vx, 'vy': vy,
            'l': l, 'w': w, 'h': h,
            'start': start_frame, 'end': end_frame
        })

    gt_sequence  = []
    det_sequence = []

    for frame in range(n_frames):
        gt_frame  = []
        det_frame = []

        for v in vehicles:
            if not (v['start'] <= frame < v['end']):
                continue

            t = frame - v['start']
            cx = v['cx'] + v['vx'] * t
            cy = v['cy'] + v['vy'] * t

            # 시야 밖(50m 이상)이면 제외
            if cy > 50 or cy < -5:
                continue

            bbox_gt = (cx, cy, v['cz'], v['theta'], v['l'], v['w'], v['h'])
            gt_frame.append((*bbox_gt, v['id']))

            # Detection noise 추가
            miss_prob = 0.05   # 5% 미검출
            fp_prob   = 0.03   # 3% FP

            if random.random() > miss_prob:
                noise_scale = 0.3
                det = (
                    cx + np.random.normal(0, noise_scale),
                    cy + np.random.normal(0, noise_scale),
                    v['cz'] + np.random.normal(0, 0.1),
                    v['theta'] + np.random.normal(0, 0.05),
                    v['l'] + np.random.normal(0, 0.1),
                    v['w'] + np.random.normal(0, 0.05),
                    v['h'] + np.random.normal(0, 0.05),
                )
                det_frame.append(det)

            if random.random() < fp_prob:
                fp = (
                    cx + random.uniform(5, 10),
                    cy + random.uniform(5, 10),
                    0.75, 0.0, 4.5, 2.0, 1.5
                )
                det_frame.append(fp)

        gt_sequence.append(gt_frame)
        det_sequence.append(det_frame)

    return gt_sequence, det_sequence


gt_seq, det_seq = generate_kitti_sequence(n_frames=60, n_vehicles=8)
print(f'시퀀스 생성 완료: {len(gt_seq)} 프레임')
print(f'평균 GT 객체수: {np.mean([len(f) for f in gt_seq]):.1f}개/프레임')
print(f'평균 Detection수: {np.mean([len(f) for f in det_seq]):.1f}개/프레임')

## 5. 추적 실행 및 MOTA/MOTP 평가

In [ ]:
def compute_mot_metrics(gt_sequence, tracked_sequence, iou_threshold=0.25):
    """
    MOTA / MOTP 계산

    MOTA = 1 - (FP + FN + IDSW) / GT_total
    MOTP = Σ IoU_matched / Σ matched_count
    """
    total_gt  = 0
    total_fp  = 0
    total_fn  = 0
    total_idsw = 0
    total_match_iou = 0.0
    total_matched   = 0

    prev_gt_to_track = {}   # gt_id → track_id (ID switch 검출용)

    for frame_idx, (gt_frame, trk_frame) in enumerate(zip(gt_sequence, tracked_sequence)):
        total_gt += len(gt_frame)

        if len(gt_frame) == 0 and len(trk_frame) == 0:
            continue

        if len(gt_frame) == 0:
            total_fp += len(trk_frame)
            continue

        if len(trk_frame) == 0:
            total_fn += len(gt_frame)
            continue

        # IoU Matrix
        n_gt  = len(gt_frame)
        n_trk = len(trk_frame)
        iou_mat = np.zeros((n_gt, n_trk))
        for gi, gt_obj in enumerate(gt_frame):
            for ti, trk_obj in enumerate(trk_frame):
                iou_mat[gi, ti] = iou_bev(gt_obj[:7], trk_obj[:7])

        # Hungarian matching
        row_ind, col_ind = linear_sum_assignment(1.0 - iou_mat)

        matched_gt  = set()
        matched_trk = set()
        curr_gt_to_track = {}

        for gi, ti in zip(row_ind, col_ind):
            if iou_mat[gi, ti] >= iou_threshold:
                matched_gt.add(gi)
                matched_trk.add(ti)
                total_match_iou += iou_mat[gi, ti]
                total_matched   += 1

                gt_id  = gt_frame[gi][-1]
                trk_id = int(trk_frame[ti][-1])
                curr_gt_to_track[gt_id] = trk_id

                # ID switch 검출
                if gt_id in prev_gt_to_track:
                    if prev_gt_to_track[gt_id] != trk_id:
                        total_idsw += 1

        total_fn += (n_gt  - len(matched_gt))
        total_fp += (n_trk - len(matched_trk))
        prev_gt_to_track = curr_gt_to_track

    mota = 1.0 - (total_fp + total_fn + total_idsw) / (total_gt + 1e-9)
    motp = total_match_iou / (total_matched + 1e-9)

    return {
        'MOTA': mota, 'MOTP': motp,
        'FP': total_fp, 'FN': total_fn, 'IDSW': total_idsw,
        'GT_total': total_gt, 'Matched': total_matched
    }


# AB3DMOT 실행
tracker = AB3DMOT(iou_threshold=0.01, max_misses=3)
tracked_sequence = []

for frame_idx, dets in enumerate(det_seq):
    results = tracker.update(dets)
    tracked_sequence.append(results)

# 평가
metrics = compute_mot_metrics(gt_seq, tracked_sequence, iou_threshold=0.1)

print('AB3DMOT 평가 결과 (합성 시퀀스, 60 프레임)')
print('=' * 45)
print(f'  MOTA:  {metrics["MOTA"]:+.4f}')
print(f'  MOTP:  {metrics["MOTP"]:.4f}  (3D IoU 기준)')
print(f'  FP:    {metrics["FP"]}')
print(f'  FN:    {metrics["FN"]}')
print(f'  IDSW:  {metrics["IDSW"]}')
print(f'  GT 총합: {metrics["GT_total"]}')
print('=' * 45)

## 6. BEV 궤적 시각화

In [ ]:
# ── BEV 궤적 시각화 ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

cmap = plt.get_cmap('tab10')

for ax, (sequence, title) in zip(axes,
        [(gt_seq, 'GT 궤적 (Bird\'s Eye View)'),
         (tracked_sequence, 'AB3DMOT 추적 궤적')]):

    # 궤적 딕셔너리 구성
    traj = defaultdict(list)
    for frame_idx, frame in enumerate(sequence):
        for obj in frame:
            cx, cy = obj[0], obj[1]
            obj_id = int(obj[-1])
            traj[obj_id].append((frame_idx, cx, cy))

    # 각 객체 궤적 그리기
    for i, (obj_id, pts) in enumerate(traj.items()):
        color = cmap(i % 10)
        xs = [p[1] for p in pts]
        ys = [p[2] for p in pts]
        ax.plot(xs, ys, '-o', color=color, markersize=3,
                linewidth=1.5, label=f'ID {obj_id}')
        # 마지막 위치에 ID 표시
        ax.annotate(f'{obj_id}', (xs[-1], ys[-1]),
                    textcoords='offset points', xytext=(4,4),
                    fontsize=8, color=color)

    ax.set_xlim(-30, 30)
    ax.set_ylim(-5, 55)
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m) — 전진 방향')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

plt.suptitle('AB3DMOT — BEV 3D 다중 객체 추적', fontsize=13)
plt.tight_layout()
plt.savefig('bev_trajectories.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2D vs 3D Tracking 비교 시각화 ───────────
fig, ax = plt.subplots(figsize=(10, 5))

comparison = [
    ('ByteTrack\n(2D)', 0.9412, 0.0, 0.85, '2D IoU'),
    ('AB3DMOT\n(3D)', metrics['MOTA'], metrics['IDSW'], metrics['MOTP'], '3D IoU'),
]

names  = [c[0] for c in comparison]
motas  = [c[1] for c in comparison]
idsws  = [c[2] for c in comparison]
motps  = [c[3] for c in comparison]

x = np.arange(len(names))
w = 0.25

bars1 = ax.bar(x - w, motas,  w, label='MOTA',       color='steelblue')
bars2 = ax.bar(x,     motps,  w, label='MOTP (IoU)', color='coral')

for bar, val in zip(bars1, motas):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)
for bar, val in zip(bars2, motps):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('2D ByteTrack vs 3D AB3DMOT 성능 비교')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('2d_vs_3d_tracking.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. 면접 Q&A

**Q. 2D Tracking과 3D Tracking의 가장 큰 차이는 무엇인가요?**

> 2D Tracking은 이미지 평면의 (x, y, w, h)로 객체를 표현하므로 깊이 정보가 없습니다. 같은 차량이 가까이 있을 때와 멀리 있을 때 bbox 크기가 달라지므로 ID 스위치가 발생하기 쉽습니다. 3D Tracking은 실제 3D 공간의 (cx, cy, cz, θ, l, w, h)로 표현하므로 **크기가 일정**하고 깊이 방향 속도도 추정할 수 있습니다. 자율주행에서는 충돌 예측, 경로 계획을 위해 3D 추적이 필수입니다.

**Q. AB3DMOT에서 3D IoU를 BEV에서 계산하는 이유는?**

> 차량은 주로 수평면에서 이동하고 높이 변화는 거의 없습니다. BEV(Bird's Eye View)에서 계산하면 회전(θ)만 고려한 2D 폴리곤 교집합으로 계산이 단순해지면서도 핵심 정보(x, y 위치, heading)를 모두 반영할 수 있습니다. 완전한 3D IoU는 계산 비용이 높고 자율주행 씬에서 실질적 차이가 크지 않습니다.

**Q. Kalman Filter에서 Joseph form 갱신을 사용하는 이유는?**

> 표준 Kalman 갱신 `P = (I-KH)P`는 수치 오류 누적 시 공분산 행렬이 양의 정치 행렬(positive definite)을 잃을 수 있습니다. Joseph form `P = (I-KH)P(I-KH)^T + KRK^T`는 항상 대칭 양의 정치 행렬을 보장하므로 실제 구현에서 더 안정적입니다.

**Q. MOTA와 MOTP의 차이를 설명해주세요.**

> MOTA(Multi-Object Tracking Accuracy)는 `1 - (FP+FN+IDSW) / GT_total`로 ID 스위치, 미검출, 오검출을 모두 반영하는 **전체 추적 품질** 지표입니다. MOTP(Multi-Object Tracking Precision)는 매칭된 쌍의 평균 IoU로 **위치 정확도**만 측정합니다. MOTA가 높아도 MOTP가 낮으면 추적은 되지만 박스 위치가 부정확하다는 의미입니다.

In [ ]:
print('=' * 60)
print('skillup_round6 / 02 3D MOT — 최종 결과')
print('=' * 60)
print(f'  모델:   AB3DMOT (3D Kalman Filter + Hungarian)')
print(f'  데이터: 합성 KITTI 포맷 시퀀스 (60 프레임, 8 차량)')
print(f'  MOTA:   {metrics["MOTA"]:+.4f}')
print(f'  MOTP:   {metrics["MOTP"]:.4f}  (3D IoU 기준)')
print(f'  ID_SW:  {metrics["IDSW"]}')
print(f'  FP:     {metrics["FP"]}   FN: {metrics["FN"]}')
print()
print('구현 핵심:')
print('  1. 3D Kalman Filter (9D 상태 벡터: 위치+방향+크기+속도)')
print('  2. BEV 회전 IoU (Sutherland-Hodgman 폴리곤 클리핑)')
print('  3. MOTA/MOTP 평가 파이프라인')
print('  4. ByteTrack(2D) vs AB3DMOT(3D) 비교')
print('=' * 60)